# 08 - model comparison: 1d cnn on mel vs mfcc + svm

**purpose**: final side-by-side comparison of the two models (no ensemble).

this notebook does **not re-run any inference on val/test metrics**. it loads the per-split JSONs saved by 06 (val) and 07 (test), adds model size and the training-set accuracy (the only inference 08 runs), and emits a single aggregated `comparison_results.json`.

**inputs** (produced by upstream notebooks):
- `data/predictions/eval_results_svm.json` (06.1) - val set metrics for mfcc + svm
- `data/predictions/eval_results_cnn.json` (06.2) - val set metrics for 1d cnn on mel
- `data/predictions/test_results_svm.json` (07.1) - test set metrics + cpu latency for svm
- `data/predictions/test_results_cnn.json` (07.2) - test set metrics + cpu latency for cnn
- `data/models/svm_best.pkl`, `data/models/mfcc_scaler.pkl` - for training-set accuracy inference
- `data/models/cnn1d_best.pth` - for training-set accuracy inference
- `data/processed/X_train_{mel,mfcc}.npy`, `y_train.npy` - for training-set accuracy inference

**output**: `data/predictions/comparison_results.json` - the single source of truth for the final report.


In [1]:
import sys
sys.path.append("../../")

**import modules**. json, numpy, pandas, sklearn metrics, and model loading helpers for both svm and cnn.

In [2]:
import json
import os
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report
from src.config.config import settings
from src.models.svm_model import load_svm, predict_svm
from src.models.cnn1d_model import load_cnn, predict_cnn

## Load results from 06 (val) and 07 (test)

In [3]:
with open(settings.PREDICTIONS_DIR / "eval_results_svm.json") as f:
    eval_svm = json.load(f)
with open(settings.PREDICTIONS_DIR / "eval_results_cnn.json") as f:
    eval_cnn = json.load(f)
with open(settings.PREDICTIONS_DIR / "test_results_svm.json") as f:
    test_svm = json.load(f)
with open(settings.PREDICTIONS_DIR / "test_results_cnn.json") as f:
    test_cnn = json.load(f)
print("loaded: eval_svm, eval_cnn, test_svm, test_cnn")

loaded: eval_svm, eval_cnn, test_svm, test_cnn


## Model sizes on disk (lightweight is a project requirement)

In [4]:
svm_path    = settings.MODELS_DIR / "svm_best.pkl"
scaler_path = settings.MODELS_DIR / "mfcc_scaler.pkl"
cnn_path    = settings.MODELS_DIR / "cnn1d_best.pth"

**print svm model size**. the svm model file is ~3mb (mostly the support vectors from the rbf kernel).

In [5]:
svm_size_kb    = os.path.getsize(svm_path) / 1024
scaler_size_kb = os.path.getsize(scaler_path) / 1024
svm_total_kb   = svm_size_kb + scaler_size_kb
print(f"mfcc + svm: model={svm_size_kb:.1f} KB + scaler={scaler_size_kb:.1f} KB = {svm_total_kb:.1f} KB")

mfcc + svm: model=3045.4 KB + scaler=6.2 KB = 3051.6 KB


**print cnn model size**. the cnn is only ~529kb - about 6x smaller than the svm.

In [6]:
cnn_size_kb    = os.path.getsize(cnn_path) / 1024
print(f"1d cnn on mel: model={cnn_size_kb:.1f} KB")

1d cnn on mel: model=529.3 KB


## Training set (actors 1-19)

training accuracy is computed in-place by running predict_svm and predict_cnn on the full training set. the train-vs-val gap is the key generalization signal: a large gap means the model memorized the training speakers rather than learning emotion-discriminative features.


In [7]:
X_train_mel  = np.load(settings.PROCESSED_DIR / "X_train_mel.npy")
X_train_mfcc = np.load(settings.PROCESSED_DIR / "X_train_mfcc.npy")
y_train      = np.load(settings.PROCESSED_DIR / "y_train.npy")

**print training data shapes**. confirm the data loaded correctly.

In [8]:
print(f"loaded train set: mel={X_train_mel.shape}, mfcc={X_train_mfcc.shape}, y={y_train.shape}")


loaded train set: mel=(1628, 128, 251), mfcc=(1628, 240), y=(1628,)


**svm training accuracy**. we run the svm on the full training set to measure in-sample accuracy.

In [9]:
svm_model, scaler = load_svm(settings.MODELS_DIR, name='svm_best')
svm_train_preds, _ = predict_svm(svm_model, X_train_mfcc, scaler)
svm_train_acc = accuracy_score(y_train, svm_train_preds)

**cnn training accuracy**. similarly compute the cnn's training accuracy.

In [10]:
cnn_model = load_cnn(settings.MODELS_DIR / "cnn1d_best.pth", device=settings.TORCH_DEVICE)
cnn_train_preds, _ = predict_cnn(cnn_model, X_train_mel, batch_size=settings.CNN_1D_BATCH_SIZE, device=settings.TORCH_DEVICE)
cnn_train_acc = accuracy_score(y_train, cnn_train_preds)

**print training set comparison table**. side-by-side: svm trains at 94.47% vs cnn at 94.90%. both fit the training data well.

In [11]:
print(f"{'train set (speakers 1-19)':<42} {'mfcc + svm':>18} {'1d cnn on mel':>18}")
print("-" * 80)
print(f"{'n samples':<42} {len(y_train):>18d} {len(y_train):>18d}")
print(f"{'train accuracy (%)':<42} {svm_train_acc*100:>17.2f}% {cnn_train_acc*100:>17.2f}%")

train set (speakers 1-19)                          mfcc + svm      1d cnn on mel
--------------------------------------------------------------------------------
n samples                                                1628               1628
train accuracy (%)                                     94.47%             94.90%


## Validation set (actors 20-22) - from 06

In [12]:
print(f"{'val set (speakers 20-22)':<40} {'mfcc + svm':>14} {'1d cnn on mel':>14}")
print("-" * 70)
print(f"{'n samples':<40} {eval_svm['n_samples']:>14d} {eval_cnn['n_samples']:>14d}")
print(f"{'accuracy (%)':<40} {eval_svm['accuracy']*100:>13.2f}% {eval_cnn['accuracy']*100:>13.2f}%")
print(f"{'parameters':<40} {'n/a (kernel)':>14} {eval_cnn.get('parameters', 'n/a'):>14}")

val set (speakers 20-22)                     mfcc + svm  1d cnn on mel
----------------------------------------------------------------------
n samples                                           264            264
accuracy (%)                                     62.88%         75.38%
parameters                                 n/a (kernel)         132806


## Test set (actors 23-24) - from 07

In [13]:
print(f"{'test set (speakers 23-24)':<42} {'mfcc + svm':>18} {'1d cnn on mel':>18}")
print("-" * 80)
print(f"{'n samples':<42} {test_svm['n_samples']:>18d} {test_cnn['n_samples']:>18d}")
print(f"{'accuracy (%)':<42} {test_svm['accuracy']*100:>17.2f}% {test_cnn['accuracy']*100:>17.2f}%")
print(f"{'model size on disk (kb)':<42} {svm_total_kb:>18.1f} {cnn_size_kb:>18.1f}")
print(f"{'parameter count':<42} {'n/a (kernel)':>18} {test_cnn.get('parameters', 'n/a'):>18}")
print()
print(f"{'cpu inference mean (ms/sample)':<42} {test_svm['cpu_latency']['mean_ms']:>18.3f} {test_cnn['cpu_latency']['mean_ms']:>18.3f}")
print(f"{'cpu inference p50 (ms/sample)':<42} {test_svm['cpu_latency']['p50_ms']:>18.3f} {test_cnn['cpu_latency']['p50_ms']:>18.3f}")
print(f"{'cpu inference p99 (ms/sample)':<42} {test_svm['cpu_latency']['p99_ms']:>18.3f} {test_cnn['cpu_latency']['p99_ms']:>18.3f}")
print(f"{'cpu throughput (samples/s)':<42} {1000.0/test_svm['cpu_latency']['mean_ms']:>18.0f} {1000.0/test_cnn['cpu_latency']['mean_ms']:>18.0f}")


test set (speakers 23-24)                          mfcc + svm      1d cnn on mel
--------------------------------------------------------------------------------
n samples                                                 176                176
accuracy (%)                                           70.45%             66.48%
model size on disk (kb)                                3051.6              529.3
parameter count                                  n/a (kernel)             132806

cpu inference mean (ms/sample)                          0.195              0.530
cpu inference p50 (ms/sample)                           0.180              0.500
cpu inference p99 (ms/sample)                           0.331              0.863
cpu throughput (samples/s)                               5125               1888


## Combined train / val / test summary 

single view of the full speaker-disjoint generalization curve. each row is a metric, each column is a split. the gap row quantifies how much accuracy each model loses from in-sample (train) to held-out speakers (val/test) - this is the overfitting story in one place.

In [14]:
print(f"{'metric':<33} {'split':<8} {'mfcc + svm':>18} {'1d cnn on mel':>18}")
print("-" * 80)
print(f"{'n samples':<33} {'train':<8} {len(y_train):>18d} {len(y_train):>18d}")
print(f"{'':<33} {'val':<8} {eval_svm['n_samples']:>18d} {eval_cnn['n_samples']:>18d}")
print(f"{'':<33} {'test':<8} {test_svm['n_samples']:>18d} {test_cnn['n_samples']:>18d}")
print()
print(f"{'accuracy':<33} {'train':<8} {svm_train_acc*100:>17.2f}% {cnn_train_acc*100:>17.2f}%")
print(f"{'':<33} {'val':<8} {eval_svm['accuracy']*100:>17.2f}% {eval_cnn['accuracy']*100:>17.2f}%")
print(f"{'':<33} {'test':<8} {test_svm['accuracy']*100:>17.2f}% {test_cnn['accuracy']*100:>17.2f}%")
print()
print(f"{'gap (train - split)':<33} {'val':<8} {(svm_train_acc - eval_svm['accuracy'])*100:>17.2f}% {(cnn_train_acc - eval_cnn['accuracy'])*100:>17.2f}%")
print(f"{'':<33} {'test':<8} {(svm_train_acc - test_svm['accuracy'])*100:>17.2f}% {(cnn_train_acc - test_cnn['accuracy'])*100:>17.2f}%")


metric                            split            mfcc + svm      1d cnn on mel
--------------------------------------------------------------------------------
n samples                         train                  1628               1628
                                  val                     264                264
                                  test                    176                176

accuracy                          train                94.47%             94.90%
                                  val                  62.88%             75.38%
                                  test                 70.45%             66.48%

gap (train - split)               val                  31.59%             19.52%
                                  test                 24.02%             28.42%


## Per-class test accuracy 


In [15]:
print(f"\n{'emotion':<12} {'svm acc':>10} {'svm support':>12} {'cnn acc':>10} {'cnn support':>12}")
for name in test_svm['emotion_names']:
    s = test_svm['per_class'][name]
    c = test_cnn['per_class'][name]
    print(f"{name:<12} {s['acc']*100:>9.1f}% {s['support']:>12d} {c['acc']*100:>9.1f}% {c['support']:>12d}")


emotion         svm acc  svm support    cnn acc  cnn support
neutral           50.0%           16      50.0%           16
calm              81.2%           32      65.6%           32
happy             62.5%           32      46.9%           32
sad               62.5%           32      75.0%           32
angry             65.6%           32      84.4%           32
fearful           90.6%           32      68.8%           32


## Save aggregated comparison results

In [16]:
comparison = {
    "val": {
        "speakers":    eval_svm['speakers'],
        "n_samples":   eval_svm['n_samples'],
        "mfcc_svm":    eval_svm,
        "cnn1d":       eval_cnn,
    },
    "test": {
        "speakers":    test_svm['speakers'],
        "n_samples":   test_svm['n_samples'],
        "mfcc_svm":    test_svm,
        "cnn1d":       test_cnn,
    },
    "model_size_kb": {
        "mfcc_svm_model":  svm_size_kb,
        "mfcc_svm_scaler": scaler_size_kb,
        "mfcc_svm_total":  svm_total_kb,
        "cnn1d":           cnn_size_kb,
    },
}
print(json.dumps(comparison, indent=4))

{
    "val": {
        "speakers": [
            20,
            21,
            22
        ],
        "n_samples": 264,
        "mfcc_svm": {
            "model": "mfcc_svm",
            "split": "val",
            "n_samples": 264,
            "accuracy": 0.6287878787878788,
            "per_class": {
                "neutral": {
                    "support": 24,
                    "correct": 18,
                    "acc": 0.75
                },
                "calm": {
                    "support": 48,
                    "correct": 37,
                    "acc": 0.7708
                },
                "happy": {
                    "support": 48,
                    "correct": 30,
                    "acc": 0.625
                },
                "sad": {
                    "support": 48,
                    "correct": 30,
                    "acc": 0.625
                },
                "angry": {
                    "support": 48,
                    "correct": 31,
   

**save comparison results**. persist the aggregated comparison json.

In [ ]:
results_path = settings.PREDICTIONS_DIR / "comparison_results.json"

**save comparison results**. persist the aggregated comparison json.

In [18]:
with open(results_path, "w") as f:
    json.dump(comparison, f, indent=2)
print(f"Saved aggregated comparison to {results_path}")


Saved aggregated comparison to E:\career\projects\lightweight-speech-emotion-recognition-on-open-datasets\data\predictions\comparison_results.json
